# 第六课｜什么是 RTL？

上一课已经能把判断写成 Boolean 关系。现在需要描述：有哪些输入输出、哪个值要保存、在什么时刻更新。

今天只解决：
> **怎样用代码一样的文字描述数字硬件，而不是描述 CPU 要逐行执行的程序？**

主要新概念：**寄存器传输级（Register-Transfer Level, RTL）**。


## 1. 概念账本

**已经知道：** Boolean logic、register、clock、clock edge。

**今天学习：** RTL、HDL、SystemVerilog、module、port。

**只预告：** `always_ff` 会先出现一次帮助阅读；下一课才要求你把 combinational path 与 sequential update 组合起来。testbench 与 waveform 留到第八课。


## 2. 四个词逐个认识

**硬件描述语言（Hardware Description Language, HDL）**：描述数字硬件结构和行为的一类语言。

**SystemVerilog**：本项目采用的 HDL 与验证语言。

**RTL**：关注寄存器保存什么，以及数据如何在时钟周期之间计算和传递。

**module / port**：module 是有明确边界的硬件单元；port 是穿过这个边界的输入输出信号。


## 3. HDL 看起来像程序，但含义不同

Python 的相邻语句通常表示先后执行。RTL 的许多语句是在描述同时存在的硬件关系。读 RTL 时先问：输入是什么？输出是什么？哪个值是 state？哪个 clock edge 更新 state？


## 4. 只读一个已知行为：clocked accumulator

它和第四课的 accumulator 是同一个规则，只是换成 SystemVerilog：

```systemverilog
module clocked_accumulator (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_value,
    output logic signed [7:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n)
            state <= '0;
        else
            state <= state + input_value;
    end
endmodule
```


## 5. 第一次读 SystemVerilog 语法

`input` / `output` 是 port 方向；`logic signed [7:0]` 表示一个 8-bit 有符号信号。

`always_ff @(posedge clk)` 表示“这里描述的是在时钟上升沿更新的寄存器行为”。本课只要求会读，不要求独立写。

`<=` 是时序 RTL 常用的 **非阻塞赋值（nonblocking assignment）**；先把它理解为“这个 edge 要写入 register 的新值”。

还有一个本课只标记、不展开的新事实：`state` 和 `input_value` 都是 8-bit signed，所以 `state + input_value` 超出 `-128..127` 时会按有限位宽 bit pattern 环绕。这里把它当作教学 artifact 的已知限制，不把它当成正式数值规范；下一课会用一个可选实验观察这种现象，正式 overflow policy 仍由 RMD-002 / RMD-003 冻结。


## 6. reset 也是 contract

`rst_n` 中 `_n` 表示 active-low：值为 0 时 reset 有效。这里是同步 reset——它只在 clock edge 上改变 state。reset polarity 与 timing 必须写清楚。


## 7. Run：只做 compile check，不提前学习 simulation

如果机器安装了 Icarus Verilog，下面只检查 SystemVerilog 能否被编译/展开；**不运行 testbench，不验证行为**。这是刻意保留给第八课的边界。


In [ ]:
from pathlib import Path
import shutil, subprocess

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists():
            return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root()
iverilog = shutil.which('iverilog')
if not iverilog:
    print('Icarus Verilog not found; RTL compile check did not run.')
else:
    subprocess.run([iverilog, '-g2012', '-tnull', str(root/'rtl/learning/clocked_accumulator.sv')], check=True)
    print('COMPILE PASS: clocked_accumulator.sv')


## 8. Observe

`COMPILE PASS` 只说明语法和基本 elaboration 能通过，不说明 `1,2,3` 一定得到 `1,3,6`。这正好引出以后为什么需要 testbench。


## 9. Try It

不运行仿真，先在代码上圈出：两个 input port、一个 output port、保存 state 的信号、clock edge、reset 分支。


## 作业

[第 6 课作业：把一个 RTL 时钟沿翻译成 Python 语义](../../exercises/zh/06_what_is_rtl.ipynb)

这份独立作业检查本课的语义理解；课程中的真实 RTL / testbench 实验仍然保留。

## 10. AI Task

让 AI 只给现有 module 加注释，标出 module boundary、ports、stored state、clocked update；不改代码。


## 11. Human Check

不用 AI，你应该能解释 HDL 与普通软件的主要语义差别；module/port 各自解决什么问题；`always_ff @(posedge clk)` 告诉你什么；为什么 compile pass 不等于 behavior correct。


## 12. Engineering Handoff

`rtl/learning/clocked_accumulator.sv` 是教学 artifact。下一课把已知 neuron contract 拆成 combinational path + sequential register update。


## 13. 项目追踪 Project Trace

- Lesson: `LSN-006`
- Prepares: `RMD-004`
- Teaching RTL: `rtl/learning/clocked_accumulator.sv`


## 14. Exit Ticket

你能解释 RTL / HDL / SystemVerilog / module / port，能读出一个最小 clocked module 的 state 和更新时刻，同时不会把“编译成功”误解成“功能验证通过”。
